# Zero-shot Qwen on Video Folder (Train/Val/Test)

Notebook này chạy **zero-shot** với Qwen-VL, đầu vào là thư mục video và vẫn dùng chia `train/val/test` như pipeline hiện tại.

- Sample đều `16` frame/video.
- Đọc câu hỏi/candidates từ `text.json`, ground-truth từ `answer.json`.
- Đánh giá accuracy theo từng split.


In [ ]:
# CELL 1: Cài thư viện (chạy 1 lần)
!pip install -q unsloth transformers accelerate decord opencv-python pillow pandas tqdm

In [ ]:
# CELL 2: Config đường dẫn
import os
from pathlib import Path

# ====== Đổi các path này theo máy/Kaggle của bạn ======
VIDEO_ROOT = Path('/kaggle/input/causalvid-videos')          # thư mục chứa video files
ANNOTATION_ROOT = Path('/kaggle/input/text-annotation/QA')   # mỗi video_id có text.json + answer.json
SPLIT_DIR = Path('/kaggle/input/casual-vid-data-split/split') # train/valid/test(.pkl hoặc .txt)

# model qwen thuần cho video
QWEN_MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# zero-shot settings
NUM_FRAMES = 16
MAX_NEW_TOKENS = 8

# ====== SAMPLE RANGE: Chạy 2 lần với list slicing ======
# Lần 1: SAMPLE_RANGE = [0, 2500]
# Lần 2: SAMPLE_RANGE = [2500, None] (None = tất cả)
SAMPLE_RANGE = [0, 2500]   # [start, end] | None = [0, None] (chạy hết)

RUN_SPLITS = ['test']          # chỉ chạy test: ['test']
DEVICE = 'cuda' if os.environ.get('CUDA_VISIBLE_DEVICES', '') != '' or os.path.exists('/proc/driver/nvidia/version') else 'cpu'

print('VIDEO_ROOT =', VIDEO_ROOT)
print('ANNOTATION_ROOT =', ANNOTATION_ROOT)
print('SPLIT_DIR =', SPLIT_DIR)
print('SAMPLE_RANGE =', SAMPLE_RANGE)
print('RUN_SPLITS =', RUN_SPLITS)
print('DEVICE =', DEVICE)


In [ ]:
# CELL 3: Imports + helper split/annotation
import json
import pickle
import random
from typing import Dict, List

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

QUESTION_TYPES = ['descriptive', 'explanatory', 'predictive', 'counterfactual']

def _read_split_ids(split_name: str, split_dir: Path) -> List[str]:
    # val có thể lưu là valid
    name_candidates = [split_name]
    if split_name == 'val':
        name_candidates.append('valid')

    for n in name_candidates:
        pkl_path = split_dir / f'{n}.pkl'
        txt_path = split_dir / f'{n}.txt'

        if pkl_path.exists():
            with open(pkl_path, 'rb') as f:
                data = pickle.load(f)
            if isinstance(data, dict):
                return sorted([str(k) for k in data.keys()])
            return sorted([str(x) for x in data])

        if txt_path.exists():
            lines = [x.strip() for x in txt_path.read_text(encoding='utf-8').splitlines() if x.strip()]
            return sorted(lines)

    raise FileNotFoundError(f'Cannot find split file for {split_name} in {split_dir}')

def build_split_records(split_name: str) -> List[Dict]:
    video_ids = _read_split_ids(split_name, SPLIT_DIR)
    records = []
    for vid in tqdm(video_ids, desc=f'Build records [{split_name}]'):
        qa_dir = ANNOTATION_ROOT / vid
        text_path = qa_dir / 'text.json'
        answer_path = qa_dir / 'answer.json'
        if not text_path.exists() or not answer_path.exists():
            continue

        try:
            text_data = json.loads(text_path.read_text(encoding='utf-8'))
            ans_data = json.loads(answer_path.read_text(encoding='utf-8'))
        except Exception:
            continue

        for qtype in QUESTION_TYPES:
            if qtype not in text_data or qtype not in ans_data:
                continue

            q_obj = text_data[qtype]
            a_obj = ans_data[qtype]
            if 'question' in q_obj and 'answer' in q_obj and 'answer' in a_obj:
                records.append({
                    'split': split_name,
                    'video_id': vid,
                    'qtype': qtype,
                    'question': q_obj['question'],
                    'candidates': q_obj['answer'],
                    'gt': int(a_obj['answer'])
                })

            if qtype in ['predictive', 'counterfactual'] and 'reason' in q_obj and 'reason' in a_obj and 'question' in q_obj:
                records.append({
                    'split': split_name,
                    'video_id': vid,
                    'qtype': f'{qtype}_reason',
                    'question': q_obj['question'],
                    'candidates': q_obj['reason'],
                    'gt': int(a_obj['reason'])
                })
    return records

In [ ]:
# CELL 4: Tìm file video + sample 16 frames đều nhau
VIDEO_EXTS = ['.mp4', '.avi', '.mov', '.mkv', '.webm']

def find_video_path(video_id: str) -> Path:
    # thử dạng root/video_id.ext
    for ext in VIDEO_EXTS:
        p = VIDEO_ROOT / f'{video_id}{ext}'
        if p.exists():
            return p
    # fallback: tìm recursive file có stem == video_id
    for ext in VIDEO_EXTS:
        matches = list(VIDEO_ROOT.rglob(f'{video_id}{ext}'))
        if matches:
            return matches[0]
    return None

def sample_uniform_frames(video_path: Path, num_frames: int = 16):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []

    indices = np.linspace(0, max(total - 1, 0), num_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame))

    cap.release()
    return frames

In [ ]:
# CELL 5: Load Qwen-VL (Unsloth, 4-bit)
import torch
from unsloth import FastVisionModel

# Unsloth loads Qwen-VL in 4-bit to save VRAM
model, processor = FastVisionModel.from_pretrained(
    model_name=QWEN_MODEL_ID,
    load_in_4bit=True,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
FastVisionModel.for_inference(model)

if not torch.cuda.is_available():
    model = model.to('cpu')

model.eval()
print('Loaded model (Unsloth):', QWEN_MODEL_ID)

In [ ]:
# CELL 6: Prompt + suy luận zero-shot
import re

def build_prompt(question: str, candidates: List[str]) -> str:
    lines = [
        'You are a video question answering assistant.',
        'Watch the video frames and choose the best option index from 0 to 4.',
        f'Question: {question}',
        'Options:'
    ]
    for i, c in enumerate(candidates[:5]):
        lines.append(f'{i}. {c}')
    lines.append('Answer with one digit only: 0, 1, 2, 3, or 4.')
    return '\n'.join(lines)

def parse_digit(text: str):
    m = re.search(r'[0-4]', text)
    return int(m.group(0)) if m else None

@torch.inference_mode()
def predict_one(record: Dict):
    video_path = find_video_path(record['video_id'])
    if video_path is None:
        return None, 'video_not_found'

    frames = sample_uniform_frames(video_path, NUM_FRAMES)
    if len(frames) == 0:
        return None, 'no_frames'

    prompt = build_prompt(record['question'], record['candidates'])
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'video', 'video': frames},
                {'type': 'text', 'text': prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], videos=[frames], return_tensors='pt')

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Greedy decoding to avoid invalid probability tensors
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False
    )
    out_text = processor.batch_decode(generated_ids[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()
    pred = parse_digit(out_text)
    return pred, out_text

In [ ]:
# CELL 7: Evaluate theo split + lưu summary metrics (count correct/total per qtype)
def evaluate_split(split_name: str, sample_range=None):
    """
    Evaluate split và lưu summary: correct_count và total_count cho mỗi qtype.
    
    :param split_name: 'train', 'val', 'test'
    :param sample_range: [start, end] để slice records, None = tất cả
    :return: summary_metrics dict
    """
    records = build_split_records(split_name)
    
    # Apply sample range [start:end]
    if sample_range is not None:
        start, end = sample_range
        records = records[start:end]
    
    print(f"Evaluating {len(records)} records from [{sample_range if sample_range else 'all'}]")
    
    # Tracking metrics
    group_acc = {'descriptive': 0, 'explanatory': 0, 
                 'predictive': 0, 'predictive_reason': 0,
                 'counterfactual': 0, 'counterfactual_reason': 0}
    group_cnt = {'descriptive': 0, 'explanatory': 0,
                 'predictive': 0, 'predictive_reason': 0,
                 'counterfactual': 0, 'counterfactual_reason': 0}
    
    pred_combined = 0
    pred_combined_cnt = 0
    cf_combined = 0
    cf_combined_cnt = 0
    
    records_by_vid = {}
    
    # Process predictions
    for rec in tqdm(records, desc=f'Zero-shot [{split_name}]'):
        pred, raw_out = predict_one(rec)
        is_correct = (pred == rec['gt']) if pred is not None else False
        
        qtype = rec['qtype']
        vid = rec['video_id']
        
        if vid not in records_by_vid:
            records_by_vid[vid] = {}
        records_by_vid[vid][qtype] = {'correct': is_correct}
        
        if qtype in group_cnt:
            group_cnt[qtype] += 1
            if is_correct:
                group_acc[qtype] += 1
    
    # Calculate combined metrics
    for vid, qtypes_dict in records_by_vid.items():
        if 'predictive' in qtypes_dict and 'predictive_reason' in qtypes_dict:
            pred_combined_cnt += 1
            if qtypes_dict['predictive']['correct'] and qtypes_dict['predictive_reason']['correct']:
                pred_combined += 1
        
        if 'counterfactual' in qtypes_dict and 'counterfactual_reason' in qtypes_dict:
            cf_combined_cnt += 1
            if qtypes_dict['counterfactual']['correct'] and qtypes_dict['counterfactual_reason']['correct']:
                cf_combined += 1
    
    # Calculate accuracy percentages
    acc_dict = {}
    for qtype in group_cnt:
        if group_cnt[qtype] > 0:
            acc_dict[qtype] = (group_acc[qtype] * 100.0 / group_cnt[qtype])
        else:
            acc_dict[qtype] = 0.0
    
    acc_par = (pred_combined * 100.0 / max(pred_combined_cnt, 1))
    acc_car = (cf_combined * 100.0 / max(cf_combined_cnt, 1))
    
    acc_d = acc_dict['descriptive']
    acc_e = acc_dict['explanatory']
    overall_acc = (acc_d + acc_e + acc_par + acc_car) / 4.0
    
    # Print results
    print("\n" + "="*80)
    print(f"Split: {split_name.upper()}")
    print("="*80)
    print(f"{'Type':<20} {'Correct':>10} {'Total':>10} {'Accuracy':>12}")
    print("-"*80)
    
    print(f"{'Acc@D':<20} {group_acc['descriptive']:>10} {group_cnt['descriptive']:>10} {acc_dict['descriptive']:>11.2f}%")
    print(f"{'Acc@E':<20} {group_acc['explanatory']:>10} {group_cnt['explanatory']:>10} {acc_dict['explanatory']:>11.2f}%")
    print(f"{'Acc@PA':<20} {group_acc['predictive']:>10} {group_cnt['predictive']:>10} {acc_dict['predictive']:>11.2f}%")
    print(f"{'Acc@PR':<20} {group_acc['predictive_reason']:>10} {group_cnt['predictive_reason']:>10} {acc_dict['predictive_reason']:>11.2f}%")
    print(f"{'Acc@CA':<20} {group_acc['counterfactual']:>10} {group_cnt['counterfactual']:>10} {acc_dict['counterfactual']:>11.2f}%")
    print(f"{'Acc@CR':<20} {group_acc['counterfactual_reason']:>10} {group_cnt['counterfactual_reason']:>10} {acc_dict['counterfactual_reason']:>11.2f}%")
    
    print("-"*80)
    print(f"{'Acc@PAR':<20} {pred_combined:>10} {pred_combined_cnt:>10} {acc_par:>11.2f}%")
    print(f"{'Acc@CAR':<20} {cf_combined:>10} {cf_combined_cnt:>10} {acc_car:>11.2f}%")
    
    print("-"*80)
    print(f"{'Acc@ALL':<20} {'-':>10} {'-':>10} {overall_acc:>11.2f}%")
    print("="*80 + "\n")
    
    # Return summary only (no detailed rows)
    return {
        'split': split_name,
        'descriptive_correct': int(group_acc['descriptive']),
        'descriptive_total': int(group_cnt['descriptive']),
        'explanatory_correct': int(group_acc['explanatory']),
        'explanatory_total': int(group_cnt['explanatory']),
        'predictive_correct': int(group_acc['predictive']),
        'predictive_total': int(group_cnt['predictive']),
        'predictive_reason_correct': int(group_acc['predictive_reason']),
        'predictive_reason_total': int(group_cnt['predictive_reason']),
        'counterfactual_correct': int(group_acc['counterfactual']),
        'counterfactual_total': int(group_cnt['counterfactual']),
        'counterfactual_reason_correct': int(group_acc['counterfactual_reason']),
        'counterfactual_reason_total': int(group_cnt['counterfactual_reason']),
        'par_correct': int(pred_combined),
        'par_total': int(pred_combined_cnt),
        'car_correct': int(cf_combined),
        'car_total': int(cf_combined_cnt),
    }


In [ ]:
# CELL 8: Chạy zero-shot + lưu summary metrics
out_dir = Path('./zero_shot_outputs')
out_dir.mkdir(parents=True, exist_ok=True)

CSV_OUTPUT = out_dir / 'metrics.csv'

summary_list = []
for split_name in RUN_SPLITS:
    summary = evaluate_split(split_name, sample_range=SAMPLE_RANGE)
    summary_list.append(summary)

summary_df = pd.DataFrame(summary_list)

print('\n' + '='*100)
print('=== SUMMARY METRICS ===')
print('='*100)
print(summary_df.to_string())
print('='*100 + '\n')

# Lưu CSV
summary_df.to_csv(str(CSV_OUTPUT), index=False)
print(f'Saved metrics to: {CSV_OUTPUT}')


In [ ]:
# CELL 9: Xem 1 mẫu + số frame + đáp án dự đoán
import random
import torch

split_name = RUN_SPLITS[0] if RUN_SPLITS else 'test'
records = build_split_records(split_name)

if len(records) == 0:
    print('No records found for split:', split_name)
else:
    max_tries = 3
    last_error = None
    for attempt in range(1, max_tries + 1):
        rec = random.choice(records)
        video_path = find_video_path(rec['video_id'])
        frames = sample_uniform_frames(video_path, NUM_FRAMES) if video_path else []
        try:
            pred, raw_out = predict_one(rec)
            print('Split:', split_name)
            print('Video ID:', rec['video_id'])
            print('QType:', rec['qtype'])
            print('Question:', rec['question'])
            print('Num frames:', len(frames))
            print('GT index:', rec['gt'])
            print('Pred index:', pred)
            print('Raw output:', raw_out)

            if rec['candidates']:
                print('GT answer:', rec['candidates'][rec['gt']])
                if pred is not None and 0 <= pred < len(rec['candidates']):
                    print('Pred answer:', rec['candidates'][pred])
            last_error = None
            break
        except Exception as exc:
            last_error = exc
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f'Attempt {attempt}/{max_tries} failed: {exc}')
    if last_error is not None:
        print('All attempts failed. Consider reducing NUM_FRAMES or MAX_NEW_TOKENS.')

In [ ]:
# CELL 10: Merge 2 metrics CSV từ 2 lần chạy + tính metrics tổng
def merge_and_recompute_metrics(csv_path1: str, csv_path2: str, output_merged_csv: str = None):
    """
    Merge 2 summary CSV files từ 2 lần chạy.
    Cộng correct_count và total_count cho mỗi qtype.
    Tính lại accuracy tổng.
    
    :param csv_path1: Đường dẫn CSV lần 1 (ví dụ [0:2500])
    :param csv_path2: Đường dẫn CSV lần 2 (ví dụ [2500:])
    :param output_merged_csv: Nơi lưu CSV merged
    """
    
    # Load 2 CSV
    df1 = pd.read_csv(csv_path1)
    df2 = pd.read_csv(csv_path2)
    
    print(f"CSV 1 ({csv_path1}): {len(df1)} rows")
    print(f"CSV 2 ({csv_path2}): {len(df2)} rows")
    
    # Merge: cộng tất cả correct_* và total_*
    merged = pd.DataFrame()
    merged['split'] = [df1.iloc[0]['split']]
    
    for col in df1.columns:
        if 'correct' in col or 'total' in col:
            merged[col] = [int(df1[col].sum()) + int(df2[col].sum())]
        elif col != 'split':
            merged[col] = df1[col].values
    
    # Tính lại accuracy từ merged counts
    def calc_accuracy(correct, total):
        return (correct * 100.0 / total) if total > 0 else 0.0
    
    acc_d = calc_accuracy(merged['descriptive_correct'].values[0], merged['descriptive_total'].values[0])
    acc_e = calc_accuracy(merged['explanatory_correct'].values[0], merged['explanatory_total'].values[0])
    acc_pa = calc_accuracy(merged['predictive_correct'].values[0], merged['predictive_total'].values[0])
    acc_pr = calc_accuracy(merged['predictive_reason_correct'].values[0], merged['predictive_reason_total'].values[0])
    acc_ca = calc_accuracy(merged['counterfactual_correct'].values[0], merged['counterfactual_total'].values[0])
    acc_cr = calc_accuracy(merged['counterfactual_reason_correct'].values[0], merged['counterfactual_reason_total'].values[0])
    acc_par = calc_accuracy(merged['par_correct'].values[0], merged['par_total'].values[0])
    acc_car = calc_accuracy(merged['car_correct'].values[0], merged['car_total'].values[0])
    
    overall_acc = (acc_d + acc_e + acc_par + acc_car) / 4.0
    
    # Print kết quả
    print("\n" + "="*80)
    print("MERGED METRICS")
    print("="*80)
    print(f"{'Type':<20} {'Correct':>10} {'Total':>10} {'Accuracy':>12}")
    print("-"*80)
    
    print(f"{'Descriptive':<20} {int(merged['descriptive_correct'].values[0]):>10} {int(merged['descriptive_total'].values[0]):>10} {acc_d:>11.2f}%")
    print(f"{'Explanatory':<20} {int(merged['explanatory_correct'].values[0]):>10} {int(merged['explanatory_total'].values[0]):>10} {acc_e:>11.2f}%")
    print(f"{'Predictive':<20} {int(merged['predictive_correct'].values[0]):>10} {int(merged['predictive_total'].values[0]):>10} {acc_pa:>11.2f}%")
    print(f"{'Pred-Reason':<20} {int(merged['predictive_reason_correct'].values[0]):>10} {int(merged['predictive_reason_total'].values[0]):>10} {acc_pr:>11.2f}%")
    print(f"{'Counterfactual':<20} {int(merged['counterfactual_correct'].values[0]):>10} {int(merged['counterfactual_total'].values[0]):>10} {acc_ca:>11.2f}%")
    print(f"{'CF-Reason':<20} {int(merged['counterfactual_reason_correct'].values[0]):>10} {int(merged['counterfactual_reason_total'].values[0]):>10} {acc_cr:>11.2f}%")
    
    print("-"*80)
    print(f"{'PAR (both)':<20} {int(merged['par_correct'].values[0]):>10} {int(merged['par_total'].values[0]):>10} {acc_par:>11.2f}%")
    print(f"{'CAR (both)':<20} {int(merged['car_correct'].values[0]):>10} {int(merged['car_total'].values[0]):>10} {acc_car:>11.2f}%")
    
    print("-"*80)
    print(f"{'Acc@ALL':<20} {'-':>10} {'-':>10} {overall_acc:>11.2f}%")
    print("  (D+E+PAR+CAR)/4")
    print("="*80 + "\n")
    
    # Save merged CSV
    if output_merged_csv is None:
        output_merged_csv = str(Path(csv_path1).parent / 'metrics_merged.csv')
    
    merged.to_csv(output_merged_csv, index=False)
    print(f"Saved merged metrics to: {output_merged_csv}")
    
    return merged

# Ví dụ:
# merged_df = merge_and_recompute_metrics(
#     'zero_shot_outputs/metrics_run1.csv',
#     'zero_shot_outputs/metrics_run2.csv'
# )


In [ ]:
# CELL 11: HƯỚNG DẪN sử dụng Sample Range + Merge
print("""
=== HƯỚNG DẪN CHẠY 2 LẦN VỚI SAMPLE RANGE ===

**Lần 1 (samples 0-2500):**
1. Set ở CELL 2: SAMPLE_RANGE = [0, 2500]
2. Chạy CELL 3, 4, 5, 6, 7, 8
3. File lưu: zero_shot_outputs/metrics.csv (1 row tổng hợp)
4. Rename: metrics_run1.csv

**Lần 2 (samples 2500 đến hết):**
1. Set ở CELL 2: SAMPLE_RANGE = [2500, None]
2. Chạy lại CELL 3, 4, 5, 6, 7, 8
3. File lưu: zero_shot_outputs/metrics.csv (cách lần 1)
4. Rename: metrics_run2.csv

**Merge kết quả:**
1. Chạy CELL 10 (uncomment + sửa đường dẫn 2 file)
2. Output: metrics_merged.csv

**Ví dụ code merge:**
merged_df = merge_and_recompute_metrics(
    csv_path1='zero_shot_outputs/metrics_run1.csv',
    csv_path2='zero_shot_outputs/metrics_run2.csv',
    output_merged_csv='zero_shot_outputs/metrics_final.csv'
)

**CSV schema (minimal):**
Columns:
  - split: [test]
  - descriptive_correct: [int]
  - descriptive_total: [int]
  - explanatory_correct: [int]
  - explanatory_total: [int]
  - predictive_correct: [int]
  - predictive_total: [int]
  - predictive_reason_correct: [int]
  - predictive_reason_total: [int]
  - counterfactual_correct: [int]
  - counterfactual_total: [int]
  - counterfactual_reason_correct: [int]
  - counterfactual_reason_total: [int]
  - par_correct: [int]
  - par_total: [int]
  - car_correct: [int]
  - car_total: [int]

**Cách merge:**
- Cộng tất cả _correct và _total từ 2 CSV
- Tính lại accuracy từ tổng counts
- Kết quả có thể tính ngược lại từ correct_count/total_count
""")
